In [5]:
# 1. Install Gradio
!pip install gradio -q

import pandas as pd
import numpy as np
import tensorflow as tf
import gradio as gr
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
from tensorflow.keras.callbacks import EarlyStopping

# 2. Data Preparation (Limited to 8000 lines for speed)
df = pd.read_csv('/content/Shakespeare_data.csv')
text = " ".join(df['PlayerLine'].dropna().astype(str).values[:8000]).lower()

chars = sorted(list(set(text)))
char_to_index = {c: i for i, c in enumerate(chars)}
index_to_char = {i: c for i, c in enumerate(chars)}

maxlen = 40
step = 3
sentences, next_chars = [], []

for i in range(0, len(text) - maxlen, step):
    sentences.append(text[i: i + maxlen])
    next_chars.append(text[i + maxlen])

x = np.zeros((len(sentences), maxlen, len(chars)), dtype=np.bool_)
y = np.zeros((len(sentences), len(chars)), dtype=np.bool_)
for i, sentence in enumerate(sentences):
    for t, char in enumerate(sentence):
        x[i, t, char_to_index[char]] = 1
    y[i, char_to_index[next_chars[i]]] = 1

# 3. Model Architecture
model = Sequential([
    LSTM(128, input_shape=(maxlen, len(chars))),
    Dense(len(chars), activation='softmax')
])
model.compile(loss='categorical_crossentropy', optimizer='adam')

# 4. Training (Compressed to 5 Epochs, Mini-Output)
print("--- Training Started (5 Epochs) ---")
model.fit(x, y,
          batch_size=512,      # High batch size for faster GPU processing
          epochs=5,            # Set to 5 epochs as requested
          verbose=2)           # verbose=2 hides progress bars; only shows 1 line per epoch
print("--- Training Complete ---")

# 5. Web App Function
def generate_web_text(seed_text, length=150, temperature=0.6):
    generated = seed_text.lower()
    if len(generated) < maxlen:
        generated = (" " * (maxlen - len(generated))) + generated

    final_output = seed_text
    for i in range(length):
        x_pred = np.zeros((1, maxlen, len(chars)))
        for t, char in enumerate(generated[-maxlen:]):
            if char in char_to_index:
                x_pred[0, t, char_to_index[char]] = 1.

        preds = model.predict(x_pred, verbose=0)[0]
        preds = np.log(preds + 1e-7) / temperature
        exp_preds = np.exp(preds)
        preds = exp_preds / np.sum(exp_preds)

        next_char = index_to_char[np.random.choice(range(len(chars)), p=preds)]
        generated += next_char
        final_output += next_char

    return final_output

# 6. Launch Web UI
demo = gr.Interface(
    fn=generate_web_text,
    inputs=[gr.Textbox(label="Enter Seed", value="Shall I compare thee")],
    outputs="text",
    title="Mini-Shakespeare Bot",
    description="Trained for 5 epochs. Type a phrase to generate text."
)

demo.launch(share=True)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


--- Training Started (5 Epochs) ---
Epoch 1/5
214/214 - 73s - 339ms/step - loss: 2.8993
Epoch 2/5
214/214 - 72s - 337ms/step - loss: 2.4856
Epoch 3/5
214/214 - 71s - 331ms/step - loss: 2.3147
Epoch 4/5
214/214 - 82s - 382ms/step - loss: 2.2327
Epoch 5/5
214/214 - 72s - 336ms/step - loss: 2.1745
--- Training Complete ---
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://d94a4d2d27387b55df.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
